In [1]:
!pip install gym numpy pandas pickle-mixin stable-baselines3[extra]

In [2]:
!pip install --upgrade stable-baselines3

In [3]:
import gym
import numpy as np
import pandas as pd
import pickle 
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import LabelEncoder

In [4]:
class TaskSchedulingEnv(gym.Env):
    def __init__(self, tasks, team):
        super(TaskSchedulingEnv, self).__init__()
        self.tasks = tasks
        self.team = team

        # Define action and observation space
        # Action: Assigning tasks to team members
        self.action_space = gym.spaces.Discrete(len(team))

        # Observation: Workload, skills, and task details
         # Corrected observation space shape
        self.observation_space = gym.spaces.Box(
            low=0,
            high=1,
            shape=(4 + len(team),),  # 4 task features + len(team) for team members
            dtype=np.float32
        )

        self.current_task_idx = 0
        self.state = self._get_state()

    def _get_state(self):
        task = self.tasks[self.current_task_idx]
    
        # Only include relevant fields in the task vector
        task_vector = np.array([
            task["Skill"],        # Skill (encoded numeric value)
            task["Deadline"],     # Deadline (days remaining)
            task["Priority"],     # Priority (numeric)
            task["Duration"]      # Duration (normalized)
        ], dtype=np.float32)
    
        # Ensure the team vector is just the Workload data (normalized)
        team_vector = np.array([member['Workload'] / 20 for member in self.team], dtype=np.float32)
    
        # Ensure that the concatenated result matches the shape of the observation space
        return np.concatenate([task_vector, team_vector])


    def step(self, action):
        member = self.team[action]
        task = self.tasks[self.current_task_idx]

        # Calculate reward
        reward = 0
        if task['Skill'] in member['Skills']:
            reward += 10  # Positive reward for matching skill

        if member['Workload'] < 10:
            reward += 5  # Reward for balanced workload
        else:
            reward -= 5  # Penalty for overloading

        if task['Deadline'] >= 0:
            reward += 5  # Reward for timely task assignment
        else:
            reward -= 10  # Penalty for overdue tasks

        # Update member's workload and task index
        member['Workload'] += task['Duration']
        self.current_task_idx += 1

        # Check if all tasks are done
        done = self.current_task_idx >= len(self.tasks)

        # Update state
        self.state = self._get_state() if not done else None

        return self.state, reward, done, {}

    def reset(self):
        self.current_task_idx = 0
        for member in self.team:
            member['Workload'] = 0
        self.state = self._get_state()
        return self.state

In [5]:
# User input system
print("Select a domain:")
domains = ["Software Development", "Content Creation", "Marketing and Sales"]
for i, domain in enumerate(domains):
    print(f"{i + 1}. {domain}")
domain_choice = int(input("Enter your choice: ")) - 1
selected_domain = domains[domain_choice]

num_team_members = int(input("Enter the number of team members: "))
team = []
for i in range(num_team_members):
    name = input(f"Enter name for team member {i + 1}: ")
    skills = input(f"Enter skills for {name} (comma-separated): ").split(",")
    team.append({"Name": name, "Skills": skills, "Workload": 0})

Select a domain:
1. Software Development
2. Content Creation
3. Marketing and Sales


Enter your choice:  1
Enter the number of team members:  1
Enter name for team member 1:  Arman
Enter skills for Arman (comma-separated):  JAVA,C++,C


In [6]:
# Load dataset for the selected domain
dataset_path = "adaptive_task_scheduling_dataset.csv"
dataset = pd.read_csv(dataset_path)
tasks = dataset.sample(50).to_dict(orient="records")  # Use a sample for training

In [7]:
from sklearn.preprocessing import LabelEncoder

# Create a label encoder for skills
skill_encoder = LabelEncoder()

# Preprocess tasks for RL
def preprocess_task(task):
    return {
        "Skill": skill_encoder.fit_transform([task["Skill Requirement"]])[0],  # Encode skill
        "Deadline": (pd.to_datetime(task["Task Deadline"]) - pd.Timestamp.now()).days,
        "Priority": ["Low", "Medium", "High"].index(task["Task Priority"]),
        "Duration": task["Estimated Completion Time"] / 8,  # Normalize duration
    }

In [8]:
rl_tasks = [preprocess_task(task) for task in tasks]

In [9]:
# Create and wrap the environment
env = DummyVecEnv([lambda: TaskSchedulingEnv(rl_tasks, team)])

C:\Users\arman\anaconda3\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [10]:
# Initialize and train the PPO model
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 713  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
--------------------------------------
| time/                   |          |
|    fps                  | 467      |
|    iterations           | 2        |
|    time_elapsed         | 8        |
|    total_timesteps      | 4096     |
| train/                  |          |
|    approx_kl            | 0.0      |
|    clip_fraction        | 0        |
|    clip_range           | 0.2      |
|    entropy_loss         | 0        |
|    explained_variance   | 0.000331 |
|    learning_rate        | 0.0003   |
|    loss                 | 1.44e+03 |
|    n_updates            | 10       |
|    policy_gradient_loss | 1.64e-09 |
|    value_loss           | 2.14e+03 |
--------------------------------------
--------------------------------------
| time/                   |     

In [11]:
# Test the model
done = False
state = env.reset()
accuracy = []  # Initialize the accuracy list

while not done:
    action, _ = model.predict(state)  # Get the predicted action
    print(f"Action: {action}, Type of Action: {type(action)}")  # Print action and its type
    
    # Since action is an integer, no need to index it or cast
    state, reward, done, _ = env.step(action)
    accuracy.append(reward > 0)  # Count successful assignments
    print(f"Action: {action}, Reward: {reward}")

# Calculate accuracy
accuracy_score = float(sum(accuracy) / len(accuracy) * 100) if accuracy else 0  # Ensure it's a float
# Print the accuracy as a formatted string
print(f"Model Accuracy: {accuracy_score:.2f}%")

Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'numpy.ndarray'>
Action: [0], Reward: [10.]
Action: [0], Type of Action: <class 'num

C:\Users\arman\AppData\Local\Temp\ipykernel_21424\4542699.py:16: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  accuracy_score = float(sum(accuracy) / len(accuracy) * 100) if accuracy else 0  # Ensure it's a float


In [ ]:
import gym
import numpy as np
import pandas as pd
import pickle
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import LabelEncoder


class TaskSchedulingEnv(gym.Env):
    def __init__(self, tasks, team):
        super(TaskSchedulingEnv, self).__init__()
        self.tasks = tasks
        self.team = team

        # Define action and observation space
        self.action_space = gym.spaces.Discrete(len(team))
        self.observation_space = gym.spaces.Box(
            low=0,
            high=1,
            shape=(4 + len(team),),  # 4 task features + len(team) for team workload
            dtype=np.float32
        )

        self.current_task_idx = 0
        self.state = self._get_state()

    def _get_state(self):
        task = self.tasks[self.current_task_idx]
        task_vector = np.array([
            task["Skill"],
            task["Deadline"],
            task["Priority"],
            task["Duration"]
        ], dtype=np.float32)
        team_vector = np.array([member['Workload'] / 20 for member in self.team], dtype=np.float32)
        return np.concatenate([task_vector, team_vector])

    def step(self, action):
        member = self.team[action]
        task = self.tasks[self.current_task_idx]

        # Reward calculation
        reward = 0
        if task['Skill'] in member['Skills']:
            reward += 20  # Higher reward for matching skill
        if member['Workload'] < 10:
            reward += 10  # Balanced workload reward
        else:
            reward -= 5  # Penalty for overloading
        if task['Deadline'] > 0:
            reward += max(0, 20 - abs(task['Deadline']))  # Gradual reward for meeting deadlines
        else:
            reward -= 20  # Penalty for overdue tasks

        # Update member's workload and current task index
        member['Workload'] += task['Duration']
        self.current_task_idx += 1

        # Check if all tasks are done
        done = self.current_task_idx >= len(self.tasks)
        self.state = self._get_state() if not done else None
        return self.state, reward, done, {}

    def reset(self):
        self.current_task_idx = 0
        for member in self.team:
            member['Workload'] = 0
        self.state = self._get_state()
        return self.state


# User input system
print("Select a domain:")
domains = ["Software Development", "Content Creation", "Marketing and Sales"]
for i, domain in enumerate(domains):
    print(f"{i + 1}. {domain}")
domain_choice = int(input("Enter your choice: ")) - 1
selected_domain = domains[domain_choice]

num_team_members = int(input("Enter the number of team members: "))
team = []
for i in range(num_team_members):
    name = input(f"Enter name for team member {i + 1}: ")
    skills = input(f"Enter skills for {name} (comma-separated): ").split(",")
    team.append({"Name": name, "Skills": skills, "Workload": 0})

# Load dataset for the selected domain
dataset_path = "adaptive_task_scheduling_dataset.csv"
dataset = pd.read_csv(dataset_path)

# Create a label encoder for skills
skill_encoder = LabelEncoder()

# Preprocess tasks for RL
def preprocess_task(task):
    return {
        "Skill": skill_encoder.fit_transform([task["Skill Requirement"]])[0],
        "Deadline": (pd.to_datetime(task["Task Deadline"]) - pd.Timestamp.now()).days,
        "Priority": ["Low", "Medium", "High"].index(task["Task Priority"]),
        "Duration": task["Estimated Completion Time"] / 8,  # Normalize duration
    }

tasks = dataset.sample(50).to_dict(orient="records")  # Use a sample for training
rl_tasks = [preprocess_task(task) for task in tasks]

# Create and wrap the environment
env = DummyVecEnv([lambda: TaskSchedulingEnv(rl_tasks, team)])

# Train PPO model
model = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-4, gamma=0.99, clip_range=0.2)
model.learn(total_timesteps=50000)

# Test the model
done = False
state = env.reset()
accuracy = []

while not done:
    action, _ = model.predict(state)
    state, reward, done, _ = env.step(action)
    accuracy.append(reward > 0)

# Calculate accuracy
accuracy_score = float(sum(accuracy) / len(accuracy) * 100) if accuracy else 0
print(f"Model Accuracy: {accuracy_score:.2f}%")

# Save the model
model_path = "task_scheduler_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"Model saved to {model_path}")

Select a domain:
1. Software Development
2. Content Creation
3. Marketing and Sales


Enter your choice:  1
Enter the number of team members:  1
Enter name for team member 1:  Arman
Enter skills for Arman (comma-separated):  JAVA,HTML,C++


C:\Users\arman\anaconda3\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
-----------------------------
| time/              |      |
|    fps             | 594  |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 2048 |
-----------------------------
--------------------------------------
| time/                   |          |
|    fps                  | 409      |
|    iterations           | 2        |
|    time_elapsed         | 9        |
|    total_timesteps      | 4096     |
| train/                  |          |
|    approx_kl            | 0.0      |
|    clip_fraction        | 0        |
|    clip_range           | 0.2      |
|    entropy_loss         | 0        |
|    explained_variance   | 0.000179 |
|    learning_rate        | 0.0003   |
|    loss                 | 3.6e+03  |
|    n_updates            | 10       |
|    policy_gradient_loss | 8.73e-10 |
|    value_loss           | 4.74e+03 |
--------------------------------------
--------------------------------------
| time/                   |     

In [ ]:
# Save the model to a .pkl file
model_path = "task_scheduler_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"Model saved to {model_path}")

In [ ]:
# Interactive Testing
print("\nInteractive Testing")
while True:
    print("\nEnter details of a new task:")
    task_skill = int(input("Skill requirement (numeric): "))
    task_deadline = int(input("Deadline (days remaining): "))
    task_priority = int(input("Priority (0-Low, 1-Medium, 2-High): "))
    task_duration = float(input("Estimated Duration (hours): ")) / 8

    new_task = {
        "Skill": task_skill,
        "Deadline": task_deadline,
        "Priority": task_priority,
        "Duration": task_duration,
    }
    rl_tasks.append(new_task)
    env = DummyVecEnv([lambda: TaskSchedulingEnv(rl_tasks, team)])

    state = env.reset()
    action, _ = model.predict(state)
    assigned_member = team[action[0]]["Name"]
    print(f"Task assigned to: {assigned_member}")

    continue_testing = input("Test another task? (yes/no): ").strip().lower()
    if continue_testing != "yes":
        break